# Augmented Lagrangian (Multiplier) Methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Bayoumy/MECH559/blob/DEV/L09/L09_augmented_lagrangian_method.ipynb)
### MECH 559 - Systems Optimization - L09

Companion notebook to the "Algorithms for nonlinearly constrained optimization" lecture - fills in the numerical example the slides point to ("Applying the augmented Lagrangian method"). The augmented Lagrangian adds a *linear* multiplier term to the quadratic penalty, and updates the multiplier estimate between outer iterations - this alleviates the ill-conditioning that comes from driving the penalty parameter $r\to0$, so a *fixed* (or slowly shrinking) $r$ can be used instead.

## Learning objectives

By the end, students should be able to:

1. Implement $T(x,r,\mu)=f(x)+\mu\,g(x)+\tfrac1r P(x)$ and the multiplier update $\mu_{k+1}=\mu_k+\tfrac{2}{r_k}g(x_k)$.
2. Compare convergence speed (in outer iterations) against the plain penalty method from `L09_penalty_and_barrier_methods.ipynb` for the same $r$ schedule.
3. Confirm that $\mu_k$ converges to the exact KKT multiplier $\mu^\star$ at the solution.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

COLORS = {"AL": "#1D5FA8", "penalty": "#D1495B", "optimum": "#2A9D5C"}
print(f"NumPy {np.__version__}, SciPy ok")

NumPy 2.5.0, SciPy ok


---
## 1. Same example problem as the penalty/barrier notebook

$$
\min_{x_1,x_2}\; f(x_1,x_2) = 3x_1^2+x_2^2
\qquad\text{s.t.}\qquad
g_1(x_1,x_2) = 2-x_2-x_1 \le 0.
$$

Analytic optimum $x^\star=(0.5,1.5)$, $f^\star=3$. At the optimum, stationarity $\nabla f+\mu^\star\nabla g_1=0$ with $\nabla f(x^\star)=(3,3)$ and $\nabla g_1=(-1,-1)$ gives the exact multiplier $\mu^\star=3$.

In [2]:
def f(x):
    return 3.0 * x[0] ** 2 + x[1] ** 2

def g1(x):
    return 2.0 - x[1] - x[0]

x_star = np.array([0.5, 1.5])
mu_star = 3.0
print(f"x* = {x_star}, f* = {f(x_star):.4f}, mu* = {mu_star}")

x* = [0.5 1.5], f* = 3.0000, mu* = 3.0


---
## 2. Augmented Lagrangian iteration

$$
T(x,r,\mu) = f(x) + \mu\,g_1(x) + \frac1r P(x), \qquad P(x)=\max\{0,g_1(x)\}^2,
$$
$$
\mu_{k+1} = \mu_k + \frac{2}{r_k}\,g_1(x_k).
$$

Unlike the pure penalty method, $r$ is held **fixed** here (no need to send it to zero) - the multiplier update does the work of driving the iterates to exact feasibility.

In [3]:
def T_augmented(x, r, mu):
    return f(x) + mu * g1(x) + (1.0 / r) * max(0.0, g1(x)) ** 2

def run_augmented_lagrangian(x0, mu0=0.0, r=1.0, n_outer=8):
    x = np.array(x0, dtype=float)
    mu = mu0
    path = [x.copy()]
    mus = [mu]
    for k in range(n_outer):
        res = minimize(T_augmented, x, args=(r, mu), method="BFGS")
        x = res.x
        mu = mu + (2.0 / r) * g1(x)
        path.append(x.copy())
        mus.append(mu)
    return np.array(path), np.array(mus)

al_path, al_mus = run_augmented_lagrangian(x0=[0.0, 0.0])
print("augmented-Lagrangian sequence:")
for k, (x, mu) in enumerate(zip(al_path, al_mus)):
    print(f"  k={k}  x=({x[0]:.6f}, {x[1]:.6f})  mu={mu:.5f}  g1={g1(x): .2e}  f={f(x):.5f}")
print(f"\nfinal error to x*: {np.linalg.norm(al_path[-1]-x_star):.2e},  mu error: {abs(al_mus[-1]-mu_star):.2e}")

augmented-Lagrangian sequence:
  k=0  x=(0.000000, 0.000000)  mu=0.00000  g1= 2.00e+00  f=0.00000
  k=1  x=(0.285714, 0.857143)  mu=1.71429  g1= 8.57e-01  f=0.97959
  k=2  x=(0.408163, 1.224490)  mu=2.44898  g1= 3.67e-01  f=1.99917
  k=3  x=(0.460641, 1.381924)  mu=2.76385  g1= 1.57e-01  f=2.54629
  k=4  x=(0.483132, 1.449396)  mu=2.89879  g1= 6.75e-02  f=2.80100
  k=5  x=(0.492771, 1.478313)  mu=2.95663  g1= 2.89e-02  f=2.91388
  k=6  x=(0.496902, 1.490705)  mu=2.98141  g1= 1.24e-02  f=2.96294
  k=7  x=(0.498672, 1.496017)  mu=2.99203  g1= 5.31e-03  f=2.98409
  k=8  x=(0.499431, 1.498293)  mu=2.99659  g1= 2.28e-03  f=2.99318

final error to x*: 1.80e-03,  mu error: 3.41e-03


---
## 3. Faster than the plain penalty method, at *fixed* $r$

Re-run the pure quadratic penalty method (no multiplier term, $\mu\equiv0$) with the **same fixed** $r$ used above, and compare how quickly each reaches the optimum.

In [4]:
def T_penalty_fixed_r(x, r):
    return f(x) + (1.0 / r) * max(0.0, g1(x)) ** 2

def run_penalty_fixed_r(x0, r=1.0, n_outer=8):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for _ in range(n_outer):
        res = minimize(T_penalty_fixed_r, x, args=(r,), method="BFGS")
        x = res.x
        path.append(x.copy())
    return np.array(path)

penalty_fixed_path = run_penalty_fixed_r(x0=[0.0, 0.0], r=1.0)
errs_al = np.linalg.norm(al_path - x_star, axis=1)
errs_pen_fixed = np.linalg.norm(penalty_fixed_path - x_star, axis=1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(errs_al, "-o", color=COLORS["AL"], label="augmented Lagrangian (fixed r=1)")
ax.semilogy(errs_pen_fixed, "-s", color=COLORS["penalty"], label="plain penalty (fixed r=1, no multiplier)")
ax.set(xlabel="outer iteration $k$", ylabel=r"$\|x_k-x^\star\|$",
       title="Multiplier update removes the need to shrink $r$")
ax.legend()
plt.tight_layout()
plt.show()

print(f"augmented Lagrangian final error : {errs_al[-1]:.2e}")
print(f"plain penalty (fixed r) final error: {errs_pen_fixed[-1]:.2e}  (stalls: r never shrinks)")

augmented Lagrangian final error : 1.80e-03
plain penalty (fixed r) final error: 6.78e-01  (stalls: r never shrinks)


C:\Users\ahbay\AppData\Local\Temp\ipykernel_71220\3249184632.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Aside: clipping the multiplier estimate

The update $\mu_{k+1}=\mu_k+\tfrac{2}{r_k}g_1(x_k)$ can, for a very inactive constraint ($g_1(x_k)\ll0$), push $\mu_{k+1}$ negative - which would contradict the KKT sign requirement $\mu\ge0$ for an inequality constraint. A common refinement clips the update: $\mu_{k+1}=\max\!\big(0,\ \mu_k+\tfrac{2}{r_k}g_1(x_k)\big)$. In this example the constraint is active throughout the iteration (the unconstrained minimizer is infeasible), so $\mu_k\ge0$ the whole time and clipping changes nothing here - but it matters whenever the constraint is (or becomes) inactive.

In [5]:
def run_augmented_lagrangian_clipped(x0, mu0=0.0, r=1.0, n_outer=8):
    x = np.array(x0, dtype=float)
    mu = mu0
    mus = [mu]
    for k in range(n_outer):
        res = minimize(T_augmented, x, args=(r, mu), method="BFGS")
        x = res.x
        mu = max(0.0, mu + (2.0 / r) * g1(x))
        mus.append(mu)
    return mus

mus_clipped = run_augmented_lagrangian_clipped(x0=[0.0, 0.0])
print("unclipped mu sequence:", np.round(al_mus, 4))
print("clipped   mu sequence:", np.round(mus_clipped, 4))
print("\nIdentical here, since g1(x_k) > 0 (constraint active) at every iterate -> no clipping is triggered.")

unclipped mu sequence: [0.     1.7143 2.449  2.7638 2.8988 2.9566 2.9814 2.992  2.9966]
clipped   mu sequence: [0.     1.7143 2.449  2.7638 2.8988 2.9566 2.9814 2.992  2.9966]

Identical here, since g1(x_k) > 0 (constraint active) at every iterate -> no clipping is triggered.


---
## Takeaways

1. The augmented Lagrangian's linear term $\mu\,g(x)$ shifts the burden of enforcing feasibility from "make $r\to0$" (which causes ill-conditioning near the boundary) to "update $\mu$ towards its exact KKT value" - so $r$ can stay fixed.
2. The multiplier update $\mu_{k+1}=\mu_k+\tfrac2r g(x_k)$ is exactly what is needed to cancel the residual constraint violation left over from the previous inner solve; at convergence $\mu_k\to\mu^\star$, the true Lagrange multiplier.
3. With $r$ fixed, the plain penalty method stalls at a fixed offset from the true optimum ($r$ never shrinks, so the quadratic penalty never gets "sharp" enough) while the augmented Lagrangian converges - this is the main practical payoff of the multiplier update.